In [1]:
import numpy as np
import random

class SnakeEnv:
    def __init__(self, size=10):
        self.size = size
        self.reset()

    def reset(self):
        self.snake = [(5, 5), (5, 4), (5, 3)] # Start with length 3 to make direction clear
        self.direction = (0, 1) # Starting direction: Right
        self.spawn_food()
        self.done = False
        return self.get_state()

    def spawn_food(self):
        while True:
            self.food = (random.randint(0, self.size-1), random.randint(0, self.size-1))
            if self.food not in self.snake:
                break

    def step(self, action):
        # Action: 0: Straight, 1: Right Turn, 2: Left Turn
        clock_wise = [(0, 1), (1, 0), (0, -1), (-1, 0)]
        idx = clock_wise.index(self.direction)

        if action == 0: # Straight
            new_dir = clock_wise[idx]
        elif action == 1: # Right Turn
            new_dir = clock_wise[(idx + 1) % 4]
        else: # Left Turn
            new_dir = clock_wise[(idx - 1) % 4]

        self.direction = new_dir

        head = (self.snake[0][0] + self.direction[0],
                self.snake[0][1] + self.direction[1])

        reward = 0
        if (head in self.snake or
            head[0] < 0 or head[0] >= self.size or
            head[1] < 0 or head[1] >= self.size):
            self.done = True
            return self.get_state(), -10, True

        self.snake.insert(0, head)
        if head == self.food:
            reward = 10
            self.spawn_food()
        else:
            self.snake.pop()
            reward = -0.1

        return self.get_state(), reward, False

    def get_state(self):
        head = self.snake[0]
        dir_l = self.direction == (0, -1)
        dir_r = self.direction == (0, 1)
        dir_u = self.direction == (-1, 0)
        dir_d = self.direction == (1, 0)

        def is_unsafe(p):
            return (p[0] < 0 or p[0] >= self.size or
                    p[1] < 0 or p[1] >= self.size or
                    p in self.snake)

        p_straight = (head[0] + self.direction[0], head[1] + self.direction[1])

        # Get directions for right and left relative to current direction
        clock_wise = [(0, 1), (1, 0), (0, -1), (-1, 0)]
        idx = clock_wise.index(self.direction)
        dr = clock_wise[(idx + 1) % 4]
        dl = clock_wise[(idx - 1) % 4]

        p_right = (head[0] + dr[0], head[1] + dr[1])
        p_left = (head[0] + dl[0], head[1] + dl[1])

        state = [
            is_unsafe(p_straight), # Danger straight
            is_unsafe(p_right),    # Danger right
            is_unsafe(p_left),     # Danger left
            dir_l, dir_r, dir_u, dir_d,
            self.food[0] < head[0], # food up
            self.food[0] > head[0], # food down
            self.food[1] < head[1], # food left
            self.food[1] > head[1]  # food right
        ]
        return np.array(state, dtype=np.float32)

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

class QNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(11, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 3) # Output: [Straight, Right, Left]
        )

    def forward(self, x):
        return self.net(x)

In [3]:
from collections import deque

class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def push(self, s, a, r, s_, d):
        self.buffer.append((s, a, r, s_, d))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, s_, d = zip(*batch)
        return (np.array(s), np.array(a), np.array(r),
                np.array(s_), np.array(d))

    def __len__(self):
        return len(self.buffer)

In [6]:
env = SnakeEnv()
q_online = QNet()
q_target = QNet()
q_target.load_state_dict(q_online.state_dict())
optimizer = optim.Adam(q_online.parameters(), lr=5e-4)
buffer = ReplayBuffer()

gamma = 0.99
epsilon = 1.0
epsilon_min = 0.01
epsilon_decay = 0.9985 # Slightly slower decay for 5000 episodes
batch_size = 64
tau = 0.005

stats = {"rewards": [], "steps": [], "food": []}

print("Starting training for 5000 episodes...")

for episode in range(1, 5001):
    s = env.reset()
    total_reward, steps, food_eaten = 0, 0, 0

    for t in range(2000):
        if random.random() < epsilon:
            a = random.randint(0, 2)
        else:
            with torch.no_grad():
                q = q_online(torch.tensor(s).float())
                a = torch.argmax(q).item()

        s_, r, done = env.step(a)
        buffer.push(s, a, r, s_, done)

        if r > 5: food_eaten += 1
        s, total_reward, steps = s_, total_reward + r, steps + 1

        if len(buffer) > batch_size:
            bs, ba, br, bs_, bd = buffer.sample(batch_size)
            bs = torch.tensor(bs).float(); ba = torch.tensor(ba); br = torch.tensor(br).float()
            bs_ = torch.tensor(bs_).float(); bd = torch.tensor(bd).float()

            next_q = q_target(bs_).max(1)[0]
            target = br + gamma * next_q * (1 - bd)
            current_q = q_online(bs).gather(1, ba.unsqueeze(1)).squeeze()

            loss = nn.MSELoss()(current_q, target.detach())
            optimizer.zero_grad(); loss.backward(); optimizer.step()

            for tp, op in zip(q_target.parameters(), q_online.parameters()):
                tp.data.copy_(tau * op.data + (1.0 - tau) * tp.data)

        if done: break

    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    stats["rewards"].append(total_reward); stats["steps"].append(steps); stats["food"].append(food_eaten)

    if episode % 100 == 0:
        avg_food = np.mean(stats['food'][-100:])
        avg_reward = np.mean(stats['rewards'][-100:])
        avg_steps = np.mean(stats['steps'][-100:])
        print(f"Ep {episode:4d} | Food: {avg_food:4.1f} | Reward: {avg_reward:6.1f} | Steps: {avg_steps:5.1f} | Eps: {epsilon:.2f}")

Starting training for 5000 episodes...
Ep  100 | Food:  0.2 | Reward:  -10.3 | Steps:  21.4 | Eps: 0.86
Ep  200 | Food:  0.6 | Reward:   -6.5 | Steps:  21.8 | Eps: 0.74
Ep  300 | Food:  0.8 | Reward:   -3.6 | Steps:  21.4 | Eps: 0.64
Ep  400 | Food:  1.4 | Reward:    1.5 | Steps:  22.8 | Eps: 0.55
Ep  500 | Food:  1.3 | Reward:    1.1 | Steps:  21.3 | Eps: 0.47
Ep  600 | Food:  1.7 | Reward:    5.1 | Steps:  23.9 | Eps: 0.41
Ep  700 | Food:  2.2 | Reward:   10.0 | Steps:  28.4 | Eps: 0.35
Ep  800 | Food:  2.7 | Reward:   14.7 | Steps:  30.6 | Eps: 0.30
Ep  900 | Food:  3.5 | Reward:   22.2 | Steps:  37.3 | Eps: 0.26
Ep 1000 | Food:  3.6 | Reward:   23.2 | Steps:  35.5 | Eps: 0.22
Ep 1100 | Food:  3.6 | Reward:   23.0 | Steps:  33.4 | Eps: 0.19
Ep 1200 | Food:  4.3 | Reward:   29.9 | Steps:  40.6 | Eps: 0.17
Ep 1300 | Food:  5.6 | Reward:   41.8 | Steps:  51.0 | Eps: 0.14
Ep 1400 | Food:  6.1 | Reward:   46.2 | Steps:  54.9 | Eps: 0.12
Ep 1500 | Food:  6.6 | Reward:   50.6 | Steps:  57.

In [8]:
import time
from IPython.display import clear_output

def render_game(env):
    board = [['.' for _ in range(env.size)] for _ in range(env.size)]
    # Vẽ thức ăn
    board[env.food[0]][env.food[1]] = 'F'
    # Vẽ thân rắn
    for i, (r, c) in enumerate(env.snake):
        if i == 0:
            board[r][c] = 'H' # Head
        else:
            board[r][c] = 'S' # Body

    print(f"Score: {len(env.snake) - 3}")
    for row in board:
        print(" ".join(row))
    print("\n")

# Chạy demo 1 ván
s = env.reset()
done = False

try:
    while not done:
        clear_output(wait=True)
        render_game(env)

        # Chọn action tốt nhất từ mạng đã train
        with torch.no_grad():
            q = q_online(torch.tensor(s).float())
            a = torch.argmax(q).item()

        s, r, done = env.step(a)
        time.sleep(0.2) # Tốc độ hiển thị

    clear_output(wait=True)
    render_game(env)
    print("Game Over!")
except KeyboardInterrupt:
    print("Demo stopped.")

Score: 20
. . . . . . . . . .
. . . . . . . . . .
. . . . . . . . . .
. . . . . . . . . .
. . . . . . . S S S
. . . . . . . . F S
. . . . . . . . . S
. . . . S S S S S S
. . . . S H S S S S
. . . . S S S S S S


Game Over!
